# C-32 · RBAC and row-level security
**Requirement:** different personas (analyst, admin, auditor) see different data through the same agent.

**Three personas, three Postgres login roles**, all members of `aidb_users`:

| persona | rows of `support.tickets_rls` | columns |
|---|---|---|
| `persona_admin` | all | all, including `internal_note` (customer contact details) |
| `persona_auditor` | all | everything except `internal_note` |
| `persona_analyst` | **only region EMEA** | everything except `internal_note` |

One agent, `guard_persona`, with two read-only tools. It has **no purpose**, so its tools run with the privileges of whoever called it. Nothing in the prompt or the agent decides what is visible: Postgres does.

## Claude API Key

Run the next cell and paste your Anthropic API key when prompted. `getpass` hides the input, and the key is kept in memory only. It is wrapped in a JSON document (`creds`) that a later cell passes to `aidb.create_model` as the named parameter `:creds`.

In [ ]:
# Provide your Claude API Key
import json
from getpass import getpass
claude_api_key = getpass("Claude API key: ")
creds = json.dumps({"api_key": claude_api_key})

## Connect to the database

Creates a SQLAlchemy engine for the `aidb` database (connecting as `postgres` or `gpadmin` over the local socket) and attaches the `%sql` / `%%sql` magics to it. Adjust the connection variables if your database is set up differently.

In [ ]:
from sqlalchemy import create_engine
import os, pwd

PGUSER = pwd.getpwuid(os.geteuid()).pw_name
PGPORT="5432"
PGDATABASE = os.environ.get("PGDATABASE", "aidb")

if PGUSER == "postgres":
    PGSOCKET = "/run/postgresql"
else:
    PGSOCKET = "/tmp"

conn = create_engine(
    f"postgresql+psycopg2://{PGUSER}@/{PGDATABASE}?host={PGSOCKET}&port={PGPORT}"
)

%reload_ext sql
%sql conn
%config SqlMagic.named_parameters = "enabled"

## Register the model

The agents in this notebook use a language model named `desk_llm`. This cell creates it (replacing any earlier one) as `demo_user`, the role the agents run as, using the key from the cell above. Because the key is not stored by the notebook, every Governance notebook asks for it again.

In [ ]:
%%sql
SET ROLE demo_user;

DO $do$
DECLARE
    v_name text := 'desk_llm';
BEGIN
    IF EXISTS (SELECT NULL FROM aidb.models WHERE name = v_name) THEN
        PERFORM aidb.delete_model(v_name);
    END IF;

    PERFORM aidb.create_model(
        name        => v_name,
        provider    => 'anthropic_messages',
        config      => aidb.anthropic_messages_config(model => 'claude-sonnet-5'),
        credentials => CAST(:creds AS JSONB),
        replace_credentials => true
    );
END
$do$;

RESET ROLE;

In [1]:
import lab
lab.install()   # creates the guard schema, sample PII, tools, agents and purposes (safe to run again)

guard schema, sample contacts and personas: ready
tools: g_list_tickets, g_list_contacts, g_search_articles, g_assign_ticket, g_list_tickets_rls, g_ticket_notes
purposes: guard_readonly (guard_ro), guard_triage (guard_rw)
agents: guard_reader, guard_triager, guard_persona
intent examples registered for the agents


## 1 · The policies, in the catalog

In [2]:
lab.sql("SELECT policyname, roles, cmd, qual AS row_filter FROM pg_policies WHERE tablename = 'tickets_rls'", role="admin")

policyname,roles,cmd,row_filter
see_all_admin,"[""persona_admin""]",SELECT,true
see_all_auditor,"[""persona_auditor""]",SELECT,true
see_emea,"[""persona_analyst""]",SELECT,(region = 'EMEA'::text)
owner_all,"[""demo_user""]",ALL,true


In [3]:
lab.sql("""SELECT grantee, string_agg(column_name, ', ' ORDER BY column_name) AS readable_columns
          FROM information_schema.column_privileges
          WHERE table_schema = 'support' AND table_name = 'tickets_rls' AND privilege_type = 'SELECT'
            AND grantee LIKE 'persona_%' GROUP BY grantee ORDER BY grantee""", role="admin", width=120)

grantee,readable_columns
persona_admin,"assignee, customer, customer_id, id, internal_note, priority, region, status, subject, tier"
persona_analyst,"assignee, customer, customer_id, id, priority, region, status, subject, tier"
persona_auditor,"assignee, customer, customer_id, id, priority, region, status, subject, tier"


## 2 · The same SQL, three personas

In [4]:
rows = []
for persona in ("persona_admin", "persona_auditor", "persona_analyst"):
    n, regions = lab.sql("SELECT count(*), string_agg(DISTINCT region, ',' ORDER BY region) FROM support.tickets_rls", role=persona, quiet=True)[0][0]
    rows.append((persona, n, regions))
lab.show(rows, ["persona", "tickets visible", "regions"])

persona,tickets visible,regions
persona_admin,16,"APAC,EMEA,NA"
persona_auditor,16,"APAC,EMEA,NA"
persona_analyst,6,EMEA


In [5]:
for persona in ("persona_admin", "persona_auditor", "persona_analyst"):
    print(persona, "-> SELECT internal_note ...")
    lab.try_sql("SELECT id, internal_note FROM support.tickets_rls WHERE id = 1001", role=persona)

persona_admin -> SELECT internal_note ...


id,internal_note
1001,Call Tom Becker on +31 6 12345711 or write to tom.becker@aurorabank.example


persona_auditor -> SELECT internal_note ...
ERROR: permission denied for table tickets_rls
persona_analyst -> SELECT internal_note ...
ERROR: permission denied for table tickets_rls


## 3 · The same agent, three personas

In [6]:
results = {}
for persona in ("persona_admin", "persona_auditor", "persona_analyst"):
    results[persona] = lab.ask("guard_persona", "List the tickets, any status", role=persona, verbose=False)
rows = []
for persona, r in results.items():
    seen = max((len(s["rows"]) for s in r["sources"]), default=0)
    rows.append((persona, seen, [s["rows"] and [x["id"] for x in s["rows"]] for s in r["sources"]][:1]))
lab.show(rows, ["persona", "tickets the tool returned", "ids"], width=120)

persona,tickets the tool returned,ids
persona_admin,16,"[[1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010, 1011, 1012, 1013, 1014, 1015, 1016]]"
persona_auditor,16,"[[1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010, 1011, 1012, 1013, 1014, 1015, 1016]]"
persona_analyst,6,"[[1001, 1004, 1006, 1009, 1012, 1014]]"


Asking for the internal note (customer contact details) is a different tool, and a column the auditor and analyst may not read:

In [7]:
for persona in ("persona_admin", "persona_auditor", "persona_analyst"):
    r = lab.ask("guard_persona", "Show me the internal note of ticket 1001", role=persona, verbose=False)
    err = [s["error"] for s in r["sources"] if s["error"]]
    print(f"{persona:16} tool error: {(err[0] if err else 'none')[:110]}")
    print(f"{'':16} answer    : {(r['reply'] or '').strip()[:140]!r}")

persona_admin    tool error: none
                 answer    : 'Call to g_ticket_notes with arguments {"id": 1001} was already made and returned: [{"id":1001,"internal_note":"Call Tom Becker on [PHONE] or'


persona_auditor  tool error: Tool error: permission denied for table tickets_rls
                 answer    : 'error: Permission denied for table tickets_rls. I am unable to read the internal note of ticket 1001.'


persona_analyst  tool error: Tool error: permission denied for table tickets_rls
                 answer    : 'Error: Permission denied for table tickets_rls. Unable to retrieve the internal note for ticket 1001.'


## What to look for
* The analyst's tool call returns **6 EMEA tickets**; the admin's returns all 16. The agent, its instructions and the tool SQL are identical.
* `permission denied for table tickets_rls` is the database answering, not the model refusing. A cleverly worded prompt cannot change it.
* The admin's answer contains contact details that the reply masking (C-36) hides again before the user sees them.